# 📚 Sample Size & Power Analysis: How Many Users, How Long?
> Section 15 of the Statistics & A/B Testing curriculum  
> 통계 & A/B 테스트 커리큘럼 섹션 15

---
# 🎯 Learning Objective (학습 목표)
Today I want to learn / 오늘 배우고 싶은 것:
- [x] Explain what Minimum Detectable Effect (MDE) is and its direct tradeoff with required sample size<br>최소 탐지 가능 효과(MDE)가 무엇이며 필요 표본 크기와의 직접적인 트레이드오프 설명하기
- [x] Calculate required sample size for a conversion-rate test using statsmodels' power analysis tools<br>statsmodels의 검정력 분석 도구로 전환율 테스트에 필요한 표본 크기 계산하기
- [x] Convert a required sample size into a realistic test duration given daily traffic<br>일일 트래픽을 고려해 필요 표본 크기를 현실적인 테스트 기간으로 변환하기

---
# 🧠 Concept (개념)

## What is it? (이것은 무엇인가?)
*(Explain it in your own words.)*

**EN:** Sample size and power analysis answers the question every A/B test design (Section 14) needs before launch: "how many users do we actually need, and for how long do we need to run this?" It connects four numbers that trade off against each other — the Minimum Detectable Effect (MDE), sample size, significance level (alpha), and statistical power — fixing any three determines the fourth.

**KR:** 표본 크기와 검정력 분석은 모든 A/B 테스트 설계(섹션 14)가 출시 전에 필요로 하는 질문에 답합니다: "실제로 사용자가 몇 명 필요하며, 얼마나 오래 테스트를 돌려야 하는가?" 이는 서로 트레이드오프 관계에 있는 네 가지 숫자 -- 최소 탐지 가능 효과(MDE), 표본 크기, 유의수준(alpha), 통계적 검정력 -- 를 연결하며, 셋을 고정하면 나머지 하나가 결정됩니다.

### The four interlocking numbers (서로 맞물린 네 가지 숫자)

| Number / 숫자 | What it means / 의미 | Typical business default / 일반적인 기본값 |
|---|---|---|
| MDE | Smallest effect worth detecting<br>탐지할 가치가 있는 최소 효과 | Set by business judgment<br>비즈니스 판단으로 결정 |
| Sample size (n)<br>표본 크기 | Users needed per group<br>그룹당 필요한 사용자 수 | Calculated<br>계산됨 |
| Alpha | False-positive tolerance<br>오탐 허용 수준 | 0.05 (Section 6) |
| Power / 검정력 | 1 - false-negative rate<br>1 - 미탐율 | 0.80 (Section 6) |

## Why do we use it? (왜 사용하는가?)
*(When is it useful?)*

**EN:** Launching a test without this calculation means one of two expensive outcomes: running it far longer than necessary (wasting time and opportunity cost), or stopping too early with too few users to reliably detect the effect you actually care about — exactly the underpowered-test problem Sections 6 and 9 warned about, now given concrete numbers.

**KR:** 이 계산 없이 테스트를 출시하면 두 가지 값비싼 결과 중 하나로 이어집니다: 필요 이상으로 오래 돌리거나(시간과 기회비용 낭비), 실제로 신경 쓰는 효과를 신뢰성 있게 탐지하기에는 사용자가 너무 적은 채로 너무 일찍 멈추는 것 -- 정확히 섹션 6과 9가 경고한 검정력 부족 문제에 이제 구체적인 숫자가 붙은 것입니다.

## When is it used in Business Analytics? (비즈니스 분석에서 언제 사용되는가?)
*(Real-world use case)*

**EN:** The very first calculation after a test is designed (Section 14) and before it's built: "given our current traffic and the smallest lift worth caring about, do we have enough users to detect it in a reasonable timeframe?" — a question that can kill, delay, or reshape a test idea before any engineering effort is spent.

**KR:** 테스트가 설계된(섹션 14) 직후, 만들어지기 전에 하는 첫 계산입니다: "현재 트래픽과 신경 쓸 만한 최소 상승폭을 고려했을 때, 합리적인 기간 안에 이를 탐지할 만큼 충분한 사용자가 있는가?" -- 어떤 엔지니어링 노력도 들어가기 전에 테스트 아이디어를 중단시키거나, 지연시키거나, 재구성할 수 있는 질문입니다.

---
# 📝 Syntax (문법)

## Basic Syntax (기본 문법)

In [9]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

baseline_rate = 0.10   # current conversion rate
mde = 0.02              # smallest absolute lift worth detecting (10% -> 12%)
target_rate = baseline_rate + mde

effect_size = proportion_effectsize(target_rate, baseline_rate)
analysis = NormalIndPower()
n_required = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.80, ratio=1.0)

print(f"Effect size (Cohen's h): {effect_size:.4f}")
print(f"Required sample size per group: {n_required:.0f}")

Effect size (Cohen's h): 0.0640
Required sample size per group: 3835


## Common Variations (자주 쓰는 변형)

In [10]:
# Solve for power at a given, fixed sample size instead
# 대신 고정된 표본 크기에서 검정력을 계산
power_at_1000 = analysis.solve_power(effect_size=effect_size, nobs1=1000, alpha=0.05, ratio=1.0)
print(f"Power with only 1000 per group: {power_at_1000:.1%}")

Power with only 1000 per group: 29.9%


---
# 🧪 Small Examples (예제)

## Example 1 — Minimum Detectable Effect (MDE): The Business Tradeoff  (최소 탐지 가능 효과: 비즈니스 트레이드오프)

In [11]:
# Business question: leadership wants to detect even a tiny 0.5 percentage
# point lift in conversion rate. How much does that cost in required sample
# size compared to only caring about a 3-point lift?
# 비즈니스 질문: 경영진이 전환율의 아주 작은 0.5%p 상승까지 탐지하고
# 싶어합니다. 3%p 상승에만 신경 쓸 때와 비교하면 필요 표본 크기가
# 얼마나 더 드는가?

baseline_rate = 0.10
alpha = 0.05
power = 0.80
analysis = NormalIndPower()

mdes = [0.005, 0.01, 0.02, 0.03, 0.05]
print(f"{'MDE (pp)':>10} {'Target rate':>12} {'Required n/group':>18}")
for mde in mdes:
    target = baseline_rate + mde
    effect_size = proportion_effectsize(target, baseline_rate)
    n = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power, ratio=1.0)
    print(f"{mde * 100:>9.1f}p {target:>11.1%} {n:>18,.0f}")

print()
print("Detecting a tiny 0.5pp lift needs dramatically more users than a 3pp lift --")
print("MDE isn't a statistical nicety, it's a direct business lever on how long")
print("and how expensive a test will be.")
print("0.5%p의 작은 상승을 탐지하려면 3%p 상승보다 극적으로 많은 사용자가")
print("필요합니다 -- MDE는 통계적 세부사항이 아니라, 테스트가 얼마나 오래")
print("걸리고 얼마나 비쌀지에 대한 직접적인 비즈니스 레버입니다.")

  MDE (pp)  Target rate   Required n/group
      0.5p       10.5%             57,756
      1.0p       11.0%             14,744
      2.0p       12.0%              3,835
      3.0p       13.0%              1,768
      5.0p       15.0%                680

Detecting a tiny 0.5pp lift needs dramatically more users than a 3pp lift --
MDE isn't a statistical nicety, it's a direct business lever on how long
and how expensive a test will be.
0.5%p의 작은 상승을 탐지하려면 3%p 상승보다 극적으로 많은 사용자가
필요합니다 -- MDE는 통계적 세부사항이 아니라, 테스트가 얼마나 오래
걸리고 얼마나 비쌀지에 대한 직접적인 비즈니스 레버입니다.


## Example 2 — Sample Size Calculation for a Conversion Rate Test (전환율 테스트의 표본 크기 계산)

In [12]:
# Business question: how many users per group do we need to detect a lift
# from 8% to 9.5% conversion rate, at standard alpha/power settings?
# 비즈니스 질문: 전환율이 8%에서 9.5%로 오르는 것을 표준 alpha/검정력
# 설정으로 탐지하려면 그룹당 몇 명이 필요한가?

baseline_rate = 0.08
target_rate = 0.095
alpha = 0.05
power = 0.80

effect_size = proportion_effectsize(target_rate, baseline_rate)
analysis = NormalIndPower()
n_per_group = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power, ratio=1.0)

relative_lift = target_rate / baseline_rate - 1
print(f"Baseline: {baseline_rate:.1%}, Target: {target_rate:.1%} (relative lift: {relative_lift:.1%})")
print(f"Alpha: {alpha}, Power: {power:.0%}")
print(f"Effect size (Cohen's h): {effect_size:.4f}")
print(f"Required sample size PER GROUP: {n_per_group:.0f}")
print(f"Total sample size (both groups): {n_per_group * 2:.0f}")

Baseline: 8.0%, Target: 9.5% (relative lift: 18.8%)
Alpha: 0.05, Power: 80%
Effect size (Cohen's h): 0.0531
Required sample size PER GROUP: 5561
Total sample size (both groups): 11122


## Example 3 — Power Curves: Visualizing the Tradeoffs (검정력 곡선: 트레이드오프 시각화)

In [13]:
# Business question: how does required sample size change as we demand
# higher confidence (power) that we'll catch a real effect? And conversely,
# if traffic caps us at 2,000 users per group, what power do we actually get?
# 비즈니스 질문: 실제 효과를 잡아낼 확신(검정력)을 더 높게 요구할수록
# 필요 표본 크기는 어떻게 변하는가? 반대로 트래픽이 그룹당 2,000명으로
# 제한된다면 실제로 얻는 검정력은 얼마인가?

baseline_rate = 0.10
target_rate = 0.12
effect_size = proportion_effectsize(target_rate, baseline_rate)
analysis = NormalIndPower()

print("Required sample size as power target rises (MDE and alpha fixed):")
print(f"{'Power':>8} {'Required n/group':>18}")
for power_target in [0.70, 0.80, 0.90, 0.95, 0.99]:
    n = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=power_target, ratio=1.0)
    print(f"{power_target:>7.0%} {n:>18,.0f}")

print()
print("Achievable power as available sample size grows (MDE and alpha fixed):")
print(f"{'n/group':>10} {'Achievable power':>18}")
for n_available in [500, 1000, 2000, 4000, 8000]:
    power_achieved = analysis.solve_power(effect_size=effect_size, nobs1=n_available, alpha=0.05, ratio=1.0)
    print(f"{n_available:>10,} {power_achieved:>17.1%}")

print()
print("This is the 'power curve' relationship: going from 80% to 95% power roughly")
print("doubles the required sample -- a common source of 'why does the stats team")
print("need SO many more users just for a bit more confidence?' conversations.")
print("이것이 '검정력 곡선' 관계입니다: 검정력을 80%에서 95%로 올리면 필요 표본이")
print("대략 두 배가 됩니다 -- '통계팀은 왜 확신을 조금 더 얻으려고 그렇게 많은")
print("사용자가 더 필요한가요?'라는 대화가 흔히 나오는 이유입니다.")

Required sample size as power target rises (MDE and alpha fixed):
   Power   Required n/group
    70%              3,015
    80%              3,835
    90%              5,133
    95%              6,349
    99%              8,976

Achievable power as available sample size grows (MDE and alpha fixed):
   n/group   Achievable power
       500             17.3%
     1,000             29.9%
     2,000             52.5%
     4,000             81.6%
     8,000             98.2%

This is the 'power curve' relationship: going from 80% to 95% power roughly
doubles the required sample -- a common source of 'why does the stats team
need SO many more users just for a bit more confidence?' conversations.
이것이 '검정력 곡선' 관계입니다: 검정력을 80%에서 95%로 올리면 필요 표본이
대략 두 배가 됩니다 -- '통계팀은 왜 확신을 조금 더 얻으려고 그렇게 많은
사용자가 더 필요한가요?'라는 대화가 흔히 나오는 이유입니다.


## Example 4 — From Sample Size to Test Duration (표본 크기에서 테스트 기간으로)

In [14]:
# Business question: our site gets 1,200 relevant visitors per day, split
# 50/50 between Control and Treatment. Given the required sample size, how
# many days must this test run? What if that's too long?
# 비즈니스 질문: 사이트는 하루에 관련 방문자 1,200명을 받고, 대조군과
# 실험군에 50/50으로 나뉩니다. 필요 표본 크기를 고려하면 테스트를 며칠
# 돌려야 하는가? 너무 길다면 어떻게 하는가?

baseline_rate = 0.10
mde = 0.01  # 1 percentage point lift
target_rate = baseline_rate + mde
effect_size = proportion_effectsize(target_rate, baseline_rate)
analysis = NormalIndPower()

n_per_group = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.80, ratio=1.0)

daily_visitors = 1200
daily_per_group = daily_visitors / 2

days_required = n_per_group / daily_per_group
weeks_required = days_required / 7

print(f"Required sample size per group: {n_per_group:.0f}")
print(f"Daily visitors per group: {daily_per_group:.0f}")
print(f"Days required: {days_required:.0f} ({weeks_required:.1f} weeks)")
print()

if weeks_required > 6:
    print(f"{weeks_required:.1f} weeks is likely too long for most product timelines. Options:")
    print("  1) Increase MDE (accept only detecting a bigger lift) -- fewer users needed")
    print("  2) Find more traffic (e.g. include a related but lower-priority user segment)")
    print("  3) Accept lower power (e.g. 70% instead of 80%) -- a real, informed risk tradeoff")

    # Show option 1 in action: what if MDE were 2pp instead of 1pp?
    mde_bigger = 0.02
    target_bigger = baseline_rate + mde_bigger
    effect_size_bigger = proportion_effectsize(target_bigger, baseline_rate)
    n_bigger = analysis.solve_power(effect_size=effect_size_bigger, alpha=0.05, power=0.80, ratio=1.0)
    days_bigger = n_bigger / daily_per_group
    print(f"\n  Option 1 in action: MDE=2pp instead of 1pp -> {days_bigger:.0f} days instead of {days_required:.0f}")
else:
    print(f"{weeks_required:.1f} weeks is a reasonable test duration -- proceed with this design.")

Required sample size per group: 14744
Daily visitors per group: 600
Days required: 25 (3.5 weeks)

3.5 weeks is a reasonable test duration -- proceed with this design.


## Example 5 — Practice: Apply All Four Concepts (연습: 네 가지 개념 모두 적용하기)

Fill in every `________` blank below. The cell runs with **no errors** even if you leave blanks — it just prints a reminder instead of that part's result.

아래 `________` 빈칸을 모두 채워보세요. 빈칸을 두어도 셀은 **에러 없이** 실행되며, 그 부분의 결과 대신 안내 메시지만 출력됩니다.

In [15]:
# ============================================================
# Practice -- fill in every ________ blank, then re-run the cell
# 연습 -- 아래 ________ 빈칸을 모두 채운 뒤 셀을 다시 실행하세요
# ============================================================
BLANK = "________"

def ready(*vals):
    return all(v != BLANK for v in vals)

# ---------- Part A: MDE -- pick an MDE and see the cost ----------
baseline_a = 0.15

mde_a = 0.02 # TODO: pick an absolute MDE in decimal form, e.g. 0.02 for 2 percentage points

if ready(mde_a):
    target_a = baseline_a + mde_a
    effect_size_a = proportion_effectsize(target_a, baseline_a)
    n_a = NormalIndPower().solve_power(effect_size=effect_size_a, alpha=0.05, power=0.80, ratio=1.0)
    print(f"MDE={mde_a * 100:.1f}pp -> target={target_a:.1%} -> required n/group={n_a:.0f}")
else:
    print("Part A \u270f\ufe0f  Fill in mde_a above, then re-run.")

# ---------- Part B: sample size calculation -- a specific test scenario ----------
baseline_b = 0.05
target_b = 0.065

alpha_b = 0.05   # TODO: significance level, e.g. 0.05
power_b = 0.80   # TODO: desired power, e.g. 0.80

if ready(alpha_b, power_b):
    effect_size_b = proportion_effectsize(target_b, baseline_b)
    n_b = NormalIndPower().solve_power(effect_size=effect_size_b, alpha=alpha_b, power=power_b, ratio=1.0)
    print(f"Required sample size per group: {n_b:.0f}")
else:
    print("Part B \u270f\ufe0f  Fill in alpha_b and power_b above, then re-run.")

# ---------- Part C: power at fixed n -- what do we actually get? ----------
baseline_c = 0.20
target_c = 0.23
effect_size_c = proportion_effectsize(target_c, baseline_c)

available_n_c = 1500   # TODO: pick a sample size per group you actually have available, e.g. 1500

if ready(available_n_c):
    power_c = NormalIndPower().solve_power(effect_size=effect_size_c, nobs1=available_n_c, alpha=0.05, ratio=1.0)
    print(f"With n={available_n_c} per group, achievable power = {power_c:.1%}")
    if power_c < 0.80:
        print("Below the conventional 80% power target -- consider a bigger MDE or more traffic.")
else:
    print("Part C \u270f\ufe0f  Fill in available_n_c above, then re-run.")

# ---------- Part D: sample size to duration ----------
required_n_d = 3500  # per group, already calculated

daily_traffic_d = 800   # TODO: pick a daily traffic number for BOTH groups combined, e.g. 800

if ready(daily_traffic_d):
    daily_per_group_d = daily_traffic_d / 2
    days_d = required_n_d / daily_per_group_d
    print(f"Required: {required_n_d}/group, daily traffic: {daily_traffic_d} total")
    print(f"Days required: {days_d:.0f} ({days_d / 7:.1f} weeks)")
else:
    print("Part D \u270f\ufe0f  Fill in daily_traffic_d above, then re-run.")

MDE=2.0pp -> target=17.0% -> required n/group=5270
Required sample size per group: 3765
With n=1500 per group, achievable power = 51.6%
Below the conventional 80% power target -- consider a bigger MDE or more traffic.
Required: 3500/group, daily traffic: 800 total
Days required: 9 (1.2 weeks)


---
# ⚠️ Common Mistakes (자주 하는 실수)

### Mistake 1 — Setting MDE Too Small "Just to Be Safe" (그냥 안전하려고 MDE를 너무 작게 설정)
Wanting to detect even a tiny 0.1 percentage point lift sounds thorough, but as Example 1 showed, shrinking the MDE can multiply the required sample size many times over — "just to be safe" can mean "this test now takes 6 months instead of 2 weeks."  
아주 작은 0.1%p 상승까지 탐지하고 싶다는 것은 철저해 보이지만, 예제 1이 보여줬듯 MDE를 줄이면 필요 표본 크기가 몇 배로 늘어날 수 있습니다 -- "그냥 안전하려고"는 "이 테스트가 이제 2주 대신 6개월 걸린다"는 뜻일 수 있습니다.

✅ **Fix / 해결:** Set MDE based on the smallest lift that would actually change a business decision, not the smallest number that's theoretically detectable.

### Mistake 2 — Forgetting to Split Daily Traffic Between Groups (일별 트래픽을 그룹으로 나누는 것을 잊음)
If the site gets 1,200 visitors a day total, each group only gets about 600 (in a 50/50 split) — using the full 1,200 as "daily sample size per group" cuts the calculated test duration roughly in half, incorrectly.  
사이트가 하루에 총 1,200명의 방문자를 받는다면(50/50 분할에서) 각 그룹은 약 600명만 받습니다 -- 전체 1,200명을 "그룹당 일일 표본 크기"로 쓰면 계산된 테스트 기간이 잘못되어 대략 절반으로 줄어듭니다.

✅ **Fix / 해결:** Always divide total daily traffic by the number of groups (and account for uneven splits) before converting to test duration.

### Mistake 3 — Deciding Sample Size After the Fact ("We'll See How It Goes") (사후에 표본 크기 결정: "돌아가는 거 보고 정하죠")
Not calculating sample size in advance and instead "running it until it feels done" is functionally the same mistake as Section 14's peeking problem — without a pre-committed n, every check of the data becomes another opportunity to stop at a lucky moment.  
표본 크기를 미리 계산하지 않고 "돌아가는 것을 보고 끝내는" 것은 기능적으로 섹션 14의 엿보기 문제와 같은 실수입니다 -- 사전에 확정된 n 없이는 데이터를 확인할 때마다 운 좋은 순간에 멈출 또 다른 기회가 됩니다.

✅ **Fix / 해결:** Calculate and commit to a sample size (Example 2's approach) as part of the pre-registered design from Section 14, before any data is collected.

### Mistake 4 — Confusing Statistical Power With "How Good the Result Looks" (통계적 검정력을 "결과가 얼마나 좋아 보이는가"와 혼동)
Power is calculated BEFORE data collection, based on a hypothetical true effect size — it answers "if a real effect of this size exists, how likely am I to detect it?", not "how confident should I be in this specific result I already got."  
검정력은 데이터를 모으기 전에, 가상의 실제 효과 크기를 바탕으로 계산됩니다 -- "이만한 크기의 실제 효과가 있다면 내가 그것을 탐지할 확률은?"에 답하지, "이미 얻은 이 특정 결과를 얼마나 신뢰해야 하는가"에 답하지 않습니다.

✅ **Fix / 해결:** Treat power as a pre-launch planning number, and interpret post-launch results using the p-value and confidence interval (Sections 6-7) instead.

### Mistake 5 — Assuming a Standard "2-Week Test" Duration Guarantees Adequate Power (표준 "2주 테스트" 기간이 충분한 검정력을 보장한다고 가정)
A fixed calendar duration (like "we always run tests for 2 weeks") might correspond to plenty of power for a high-traffic page and dangerously little power for a low-traffic one — duration alone says nothing about whether the sample size behind it is adequate.
고정된 기간(항상 2주 테스트를 돌린다 등)은 트래픽이 많은 페이지에는 충분한 검정력을, 트래픽이 적은 페이지에는 위험할 정도로 부족한 검정력을 의미할 수 있습니다 -- 기간 자체는 그 뒤의 표본 크기가 충분한지에 대해 아무것도 말해주지 않습니다.

✅ **Fix / 해결:** Always convert a proposed duration back into expected sample size and check the resulting power (Example 3's approach), rather than trusting a duration convention blindly.

### Mistake 6 — Using a Means-Based Sample Size Formula for a Proportions Test (평균 기반 표본 크기 공식을 비율 검정에 사용)
Sample size calculations for comparing two MEANS (like average order value) and for comparing two PROPORTIONS (like conversion rate) use different underlying math — `proportion_effectsize()` is specifically for proportions; a continuous-outcome test needs a different effect size formula (Cohen's d).  
두 평균(평균 주문금액 등)을 비교하는 표본 크기 계산과 두 비율(전환율 등)을 비교하는 계산은 서로 다른 근본 수학을 사용합니다 -- proportion_effectsize()는 특히 비율을 위한 것이며, 연속형 결과 검정에는 다른 효과크기 공식(Cohen's d)이 필요합니다.

✅ **Fix / 해결:** Match the effect size formula to the outcome type — proportions need `proportion_effectsize`, continuous metrics need a mean-difference-based effect size (same `NormalIndPower`/`TTestIndPower` family, different effect-size input).

---
# 💡 Tips (팁)
Useful tips or shortcuts / 유용한 팁과 단축법

- `statsmodels`'s `solve_power()` is genuinely a "solve for the missing piece" function — give it any 3 of {effect_size, nobs1, alpha, power} and it computes the 4th, which is why the same function handles both "how many users do I need" and "what power do I get with the users I have."<br>statsmodels의 solve_power()는 문자 그대로 "빠진 조각을 풀어주는" 함수입니다 -- {effect_size, nobs1, alpha, power} 중 아무 3개를 주면 4번째를 계산해주므로, 같은 함수가 "몇 명이 필요한가"와 "가진 사용자로 얻는 검정력은?"을 둘 다 처리합니다.
- Always round a calculated sample size UP, and add a small buffer (5-10%) for real-world data quality issues (bots, incomplete sessions) — the formula gives a mathematical minimum, not a comfortable target.<br>계산된 표본 크기는 항상 올림하고, 실제 데이터 품질 문제(봇, 불완전한 세션)를 위해 작은 여유(5~10%)를 더하세요 -- 공식은 수학적 최솟값을 주지, 편안한 목표치를 주지 않습니다.
- A relative lift ("20% more conversions") and an absolute lift ("2 percentage points") sound similar but produce very different MDE inputs — always double-check which one a stakeholder means before calculating.<br>상대적 상승("전환 20% 증가")과 절대적 상승("2%p")은 비슷하게 들리지만 매우 다른 MDE 입력값을 만듭니다 -- 계산 전에 이해관계자가 어느 것을 의미하는지 항상 다시 확인하세요.
- When a required test duration looks impossibly long, showing the actual sample-size-vs-MDE table (like Example 1's) to stakeholders is usually more persuasive than just saying "it'll take too long" — the tradeoff becomes visible and concrete.<br>필요한 테스트 기간이 불가능할 정도로 길어 보일 때, 이해관계자에게 그냥 "너무 오래 걸린다"고 말하기보다 실제 표본 크기 대 MDE 표(예제 1처럼)를 보여주는 것이 보통 더 설득력 있습니다 -- 트레이드오프가 눈에 보이고 구체적으로 다가옵니다.
- This whole calculation should happen for EVERY test, not just big ones — a "quick, low-stakes" test run with an accidentally underpowered sample can waste just as much engineering time as a big one, for a result nobody can trust.<br>이 계산 전체는 큰 테스트뿐 아니라 모든 테스트에 대해 이루어져야 합니다 -- "빠르고 부담 없는" 테스트도 우연히 검정력이 부족한 표본으로 돌리면, 아무도 신뢰할 수 없는 결과를 위해 큰 테스트만큼의 엔지니어링 시간을 낭비할 수 있습니다.

---
# 🔗 Related Concepts (관련 개념)

```
A/B Test Design (Section 14)
   hypothesis, primary/guardrail metrics, randomization, pre-registration
        ↓
Sample Size & Power Analysis (Section 15)  ← you are here / 현재 위치
   MDE, sample size, power curves, sample size -> duration
        ↓
A/B Test Analysis & Interpretation (Section 16)
   running the test, statistical vs practical significance
        ↓
A/B Testing Pitfalls (Section 17)
```

*How is today's topic connected to other concepts?*

**EN:** This section turns Section 6's abstract "Type II error / power" concept into a concrete number you calculate before a single user is exposed to a test — power isn't just theory anymore, it's the input to a real go/no-go conversation with a product team. The MDE-vs-sample-size tradeoff from Example 1 is the same underlying math as Section 9's multiple-comparisons cost, just running in the opposite direction: there, MORE comparisons cost you error-rate control; here, a SMALLER effect you want to detect costs you sample size. Section 16 picks up once the pre-registered sample size from this section has actually been collected — turning the plan into a result.

**KR:** 이 섹션은 섹션 6의 추상적인 "2종 오류/검정력" 개념을, 단 한 명의 사용자도 테스트에 노출되기 전에 계산하는 구체적인 숫자로 바꿉니다 -- 검정력은 더 이상 이론일 뿐이 아니라, 제품팀과의 실제 진행/보류 대화의 입력값입니다. 예제 1의 MDE 대 표본 크기 트레이드오프는 섹션 9의 다중비교 비용과 같은 근본 수학이며, 방향만 반대입니다: 거기서는 더 많은 비교가 오류율 통제 비용을 치르고, 여기서는 탐지하고 싶은 더 작은 효과가 표본 크기 비용을 치릅니다. 섹션 16은 이 섹션에서 사전등록한 표본 크기가 실제로 수집된 시점부터 이어받습니다 -- 계획을 결과로 바꿉니다.

---
# 💼 Business Example (비즈니스 예시)
*How would a Business Analyst use this?*

**Scenario / 시나리오:**

**EN:** A growth team wants to test a new referral incentive. Current referral rate is 3%, and leadership considers anything below a 0.5 percentage point lift "not worth the incentive cost." The team has 4,000 daily eligible users. Walk through whether this test is even feasible on a reasonable timeline.

**KR:** 성장팀이 새로운 추천 인센티브를 테스트하고 싶어합니다. 현재 추천율은 3%이며, 경영진은 0.5%p 미만의 상승은 "인센티브 비용을 들일 가치가 없다"고 봅니다. 팀은 하루 4,000명의 자격 있는 사용자를 보유하고 있습니다. 이 테스트가 합리적인 기간 안에 가능한지 살펴봅니다.

**To-do / 할 일:**
- [x] Calculate the required sample size for the business-defined MDE<br>비즈니스가 정의한 MDE에 대한 필요 표본 크기 계산하기
- [x] Convert to a test duration given daily eligible traffic<br>일일 자격 있는 트래픽을 고려해 테스트 기간으로 변환하기
- [x] If impractical, check the achievable power at a capped, practical duration<br>비현실적이면 제한된 실용적 기간에서 달성 가능한 검정력 확인하기
- [x] Write a recommendation for the growth team<br>성장팀을 위한 제안 작성하기

In [16]:
baseline_rate = 0.03
mde = 0.005  # smallest lift worth the incentive cost
target_rate = baseline_rate + mde
alpha = 0.05
power = 0.80

effect_size = proportion_effectsize(target_rate, baseline_rate)
analysis = NormalIndPower()
n_per_group = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power, ratio=1.0)

daily_eligible = 4000
daily_per_group = daily_eligible / 2
days_required = n_per_group / daily_per_group

print(f"Baseline: {baseline_rate:.1%}, MDE: {mde * 100:.1f}pp, Target: {target_rate:.1%}")
print(f"Required sample size per group: {n_per_group:,.0f}")
print(f"Daily eligible users per group: {daily_per_group:,.0f}")
print(f"Days required: {days_required:.0f} ({days_required / 7:.1f} weeks)")
print()

if days_required / 7 > 8:
    print("This timeline is likely too long. Let's check the achievable power if we")
    print("instead cap the test at 6 weeks:")
    max_days = 42
    max_n_per_group = daily_per_group * max_days
    achievable_power = analysis.solve_power(effect_size=effect_size, nobs1=max_n_per_group, alpha=alpha, ratio=1.0)
    print(f"  At 6 weeks ({max_n_per_group:,.0f} per group): achievable power = {achievable_power:.1%}")
    print()
    print("Recommendation: either accept the lower power, extend the timeline, or raise")
    print("the MDE (only chase a bigger lift) to bring the test back to a practical duration.")
    print()
    print("제안: 낮은 검정력을 받아들이거나, 기간을 늘리거나, MDE를 높여(더 큰 상승만")
    print("추적) 테스트를 실용적인 기간으로 되돌리세요.")
else:
    print("This timeline is reasonable -- proceed with the design as planned.")

Baseline: 3.0%, MDE: 0.5pp, Target: 3.5%
Required sample size per group: 19,715
Daily eligible users per group: 2,000
Days required: 10 (1.4 weeks)

This timeline is reasonable -- proceed with the design as planned.


---
# 📝 Summary (요약)
*Write today's concept in 3~5 sentences.*

**EN:** Sample size and power analysis connects four interlocking numbers — Minimum Detectable Effect (MDE), sample size, alpha, and power — so that fixing any three lets you solve for the fourth, most commonly using a target alpha and power (Section 6's conventions) plus a business-chosen MDE to calculate exactly how many users are needed. Shrinking the MDE to detect ever-smaller effects multiplies the required sample size, often dramatically, turning "let's be thorough" into "this test now takes months instead of weeks" — a direct, quantifiable cost tied to a business judgment call, not just statistical pedantry. Converting the required sample size into calendar time (dividing daily traffic across groups) is what actually determines whether a test is feasible, and when the answer is "too long," the honest options are a bigger MDE, more traffic, or a deliberate, informed decision to accept lower power — never quietly shortening the test and hoping for the best.

**KR:** 표본 크기와 검정력 분석은 서로 맞물린 네 가지 숫자 -- 최소 탐지 가능 효과(MDE), 표본 크기, alpha, 검정력 -- 를 연결해, 셋을 고정하면 넷째를 풀 수 있게 합니다. 가장 흔하게는 목표 alpha와 검정력(섹션 6의 관행)에 비즈니스가 선택한 MDE를 더해 정확히 몇 명의 사용자가 필요한지 계산합니다. 점점 더 작은 효과를 탐지하도록 MDE를 줄이면 필요 표본 크기가 종종 극적으로 늘어나, "철저하게 하자"를 "이 테스트가 이제 몇 주가 아니라 몇 달 걸린다"로 바꿉니다 -- 단순한 통계적 꼼꼼함이 아니라 비즈니스적 판단에 직접 연결된, 정량화 가능한 비용입니다. 필요 표본 크기를 달력 시간으로 변환하는 것(일일 트래픽을 그룹으로 나누는 것)이 테스트가 실제로 가능한지를 결정하며, 답이 "너무 길다"일 때 정직한 선택지는 더 큰 MDE, 더 많은 트래픽, 또는 낮은 검정력을 받아들이는 신중하고 정보에 근거한 결정입니다 -- 조용히 테스트를 단축하고 잘 되기를 바라는 것이 아닙니다.

---
# 📌 One Sentence Summary (한 문장 요약)
Today's topic in ONE sentence. / 오늘의 주제를 한 문장으로.

> **EN:** Sample size and power analysis links MDE, sample size, alpha, and power into one calculation, so a business-chosen "smallest lift worth detecting" translates directly into how many users (and how many weeks) a test actually needs — and when that duration is impractical, the honest fix is a bigger MDE or more traffic, not a shorter test run on hope.

> **KR:** 표본 크기와 검정력 분석은 MDE, 표본 크기, alpha, 검정력을 하나의 계산으로 연결해, 비즈니스가 선택한 "탐지할 가치가 있는 최소 상승폭"이 실제로 몇 명(그리고 몇 주)이 필요한지로 직접 번역되게 합니다 -- 그 기간이 비현실적일 때 정직한 해결책은 더 큰 MDE나 더 많은 트래픽이지, 희망에 근거해 테스트를 짧게 돌리는 것이 아닙니다.

---
# ❓ Review Questions (복습 질문)

**Q1 (EN).** A team shrinks their MDE from 3 percentage points to 0.5 percentage points "to be thorough." What happens to the required sample size, and why is this a business decision, not just a statistical one?  
**Q1 (KR).** 팀이 "철저하게 하려고" MDE를 3%p에서 0.5%p로 줄입니다. 필요 표본 크기는 어떻게 되며, 이것이 왜 단순한 통계적 결정이 아니라 비즈니스적 결정인가요?

The required sample size goes up a lot. It grows roughly with 1/MDE², so going from 3 to 0.5 percentage points needs about 36 times more users. More users mean more time and cost, so the team must decide whether detecting a tiny effect is worth that.  
필요 표본 크기가 크게 늘어납니다. 대략 1/MDE²에 비례해서 3%p에서 0.5%p로 줄이면 약 36배의 사용자가 필요합니다. 사용자가 늘면 시간과 비용이 드니, 아주 작은 효과를 잡을 가치가 있는지는 비즈니스가 정해야 합니다.

**Q2 (EN).** A site gets 2,000 visitors a day, split evenly between Control and Treatment. If the required sample size is 15,000 per group, how many days should you plan for, and what mistake would double-count if you forgot to split the traffic?  
**Q2 (KR).** 사이트가 하루 2,000명의 방문자를 받고 대조군과 실험군에 균등하게 나뉩니다. 필요 표본 크기가 그룹당 15,000명이라면 며칠을 계획해야 하며, 트래픽을 나누는 것을 잊으면 어떤 실수가 이중 계산되나요?

Each group gets 1,000 visitors a day, so 15,000 ÷ 1,000 = 15 days. If you forget to split the traffic, you count all 2,000 visitors toward one group, get 15,000 ÷ 2,000 = 7.5 days, and stop at half the needed sample.  
각 그룹이 하루 1,000명씩 받으므로 15,000 ÷ 1,000 = 15일입니다. 트래픽을 나누는 것을 잊으면 2,000명 전부를 한 그룹으로 세어 15,000 ÷ 2,000 = 7.5일로 계산하고, 필요한 표본의 절반에서 멈추게 됩니다.

**Q3 (EN).** What's the difference between calculating power BEFORE a test (as planning) and trying to interpret "how confident we should be" AFTER seeing results?  
**Q3 (KR).** 테스트 '전에' 검정력을 계산하는 것(계획으로서)과 결과를 본 '후에' "얼마나 확신해야 하는가"를 해석하려는 것의 차이는 무엇인가요?

Before the test, power is a plan: the chance of detecting a real effect of a chosen size with a chosen sample size, used to set the sample size and duration. After the test, "post-hoc power" is just a restatement of the p-value and adds no new information. To judge confidence after results, use the confidence interval.  
테스트 전의 검정력은 계획입니다. 정한 크기의 실제 효과를 정한 표본 크기로 잡아낼 확률이고, 표본 크기와 기간을 정하는 데 씁니다. 결과를 본 후의 "사후 검정력"은 p-value를 다시 말한 것일 뿐 새 정보가 없습니다. 결과 후의 확신은 신뢰구간으로 판단합니다.

**Q4 (EN).** A calculated test duration comes out to 5 months, which is impractical. Name three legitimate options for addressing this, and one illegitimate one.  
**Q4 (KR).** 계산된 테스트 기간이 5개월로 나와 비현실적입니다. 이를 해결하는 정당한 방법 세 가지와, 부당한 방법 한 가지를 말해보세요.

Legitimate: (1) accept a larger MDE, (2) reduce variance or use a more sensitive metric (for example CUPED), (3) send more traffic or run on more pages. Illegitimate: peeking and stopping early when the result looks significant.  
정당한 방법은 (1) MDE를 더 크게 받아들이기, (2) 분산을 줄이거나 더 민감한 지표 쓰기(예: CUPED), (3) 더 많은 트래픽이나 더 많은 페이지에서 실행하기입니다. 부당한 방법은 결과가 유의해 보일 때 엿보고 조기 종료하는 것입니다.

**Q5 (EN).** Why does `proportion_effectsize()` specifically apply to a conversion-rate test, and what would you need instead for a test comparing average order values?  
**Q5 (KR).** `proportion_effectsize()`는 왜 특히 전환율 검정에 적용되며, 평균 주문금액을 비교하는 검정에는 대신 무엇이 필요한가요?

proportion_effectsize() computes Cohen's h, an effect size for two proportions (conversion rates), because a proportion's variance depends on its own value. For average order values you need Cohen's d, which needs the difference in means and the standard deviation.  
proportion_effectsize()는 두 비율(전환율)의 효과 크기인 Cohen's h를 계산합니다. 비율의 분산은 비율 값 자체에 따라 달라지기 때문입니다. 평균 주문금액은 평균의 차이와 표준편차가 필요한 Cohen's d를 써야 합니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*